# Assignment 2 — Module 1 & 2: R Programming Lab
**Lab 3:** Control Flow for Data Cleaning (UCI Heart Disease Dataset)

**Lab 4:** Advanced Missing Data Handling (UCI Adult / Census Income Dataset)

R code runs inside this Python (Colab) notebook via the `rpy2` R-magic extension.

In [ ]:
!pip install rpy2 -q
%load_ext rpy2.ipython

## Lab 3: Control Flow for Data Cleaning
Dataset: UCI Heart Disease Dataset — variable `trestbps` (resting blood pressure).
Tasks: if-else BP-cleaning function, `tryCatch()` error handling, loop vs vectorized comparison, and validation of the cleaned data.

In [ ]:
%%R
# =============================================================
# Lab 3: Control Flow for Data Cleaning
# Topic  : Loops, Functions and Error Handling in R
# Dataset: UCI Heart Disease Dataset (heart_disease.csv, 303 records)
# Variable of interest: trestbps (resting blood pressure, mm Hg)
# =============================================================

# -------------------------------------------------------------
# SETUP: Import the dataset and inject realistic data-entry errors
# -------------------------------------------------------------

import_dataset <- function(path) {
  tryCatch({
    data <- read.csv(path, stringsAsFactors = FALSE)
    cat("File imported successfully.\n")
    return(data)
  },
  error = function(e) {
    if (!file.exists(path)) {
      message("Error: The file was not found at the specified path.")
    } else {
      message("Error: The file could not be opened or is in an incorrect format.")
    }
    message("Details: ", conditionMessage(e))
    return(NULL)
  })
}

heart <- import_dataset("https://raw.githubusercontent.com/kb22/Heart-Disease-Prediction/master/dataset.csv")
cat("Rows:", nrow(heart), " Columns:", ncol(heart), "\n")

# Deterministic row positions used to simulate data-entry problems so the
# demonstration is fully reproducible on every run.
neg_idx     <- c(5, 47, 90, 133, 176, 219)   # will become negative BP
na_idx      <- c(12, 55, 98, 141, 184, 227)  # will become missing BP
extreme_idx <- c(20, 63, 106, 149, 192, 235) # will become BP > 300 mmHg

heart$trestbps_raw <- heart$trestbps                              # untouched copy, kept for reference
heart$trestbps[neg_idx]     <- -heart$trestbps[neg_idx]           # sign-flip -> negative BP
heart$trestbps[na_idx]      <- NA                                 # simulate missing observations
heart$trestbps[extreme_idx] <- heart$trestbps[extreme_idx] + 175  # push above 300 mmHg

cat("\n--- After injecting data-entry problems ---\n")
cat("Negative values:", sum(heart$trestbps < 0, na.rm = TRUE), "\n")
cat("Missing values :", sum(is.na(heart$trestbps)), "\n")
cat("Values > 300   :", sum(heart$trestbps > 300, na.rm = TRUE), "\n")


# -------------------------------------------------------------
# TASK 1: BP-Cleaning Function Using if-else
# -------------------------------------------------------------
# - Negative BP        -> NA
# - BP > 250 mmHg      -> capped at 250
# - Valid BP           -> unchanged

clean_bp <- function(x) {
  if (is.na(x)) {
    return(NA)
  } else if (x < 0) {
    return(NA)
  } else if (x > 250) {
    return(250)
  } else {
    return(x)
  }
}

heart$trestbps_clean <- sapply(heart$trestbps, clean_bp)

cat("\n--- Task 1: clean_bp() applied ---\n")
cat("Remaining negative after cleaning:", sum(heart$trestbps_clean < 0, na.rm = TRUE), "\n")
cat("Remaining >250 after cleaning    :", sum(heart$trestbps_clean > 250, na.rm = TRUE), "\n")
cat("Total NA after cleaning          :", sum(is.na(heart$trestbps_clean)), "\n")


# -------------------------------------------------------------
# TASK 2: Error Handling with tryCatch()
# -------------------------------------------------------------

# 2a. Safely calculate mean BP when missing values are present
safe_mean_bp <- function(x) {
  tryCatch({
    if (all(is.na(x))) stop("All BP values are missing; mean cannot be computed.")
    m <- mean(x, na.rm = TRUE)
    if (is.nan(m)) stop("Mean computation returned NaN.")
    return(m)
  },
  error = function(e) {
    message("safe_mean_bp() error: ", conditionMessage(e))
    return(NA)
  })
}

cat("\n--- Task 2: safe_mean_bp() ---\n")
cat("Safe mean BP (with NAs present):", round(safe_mean_bp(heart$trestbps_clean), 2), "\n")

# 2b. Safely calculate chol / trestbps, handling zero / NA / invalid denominators
safe_ratio <- function(numerator, denominator) {
  tryCatch({
    if (length(denominator) == 0) stop("Denominator vector is empty.")
    result <- numerator / denominator
    bad <- is.nan(result) | is.infinite(result)
    if (any(bad)) {
      warning(sprintf("%d ratio value(s) invalid (zero/NA/Inf denominator).", sum(bad)))
    }
    result[bad] <- NA
    return(result)
  },
  error = function(e) {
    message("safe_ratio() error: ", conditionMessage(e))
    return(rep(NA, length(numerator)))
  },
  warning = function(w) {
    message("safe_ratio() warning: ", conditionMessage(w))
    result <- numerator / denominator
    result[is.nan(result) | is.infinite(result)] <- NA
    return(result)
  })
}

heart$chol_bp_ratio <- safe_ratio(heart$chol, heart$trestbps_clean)
cat("chol/trestbps ratio -> NA count:", sum(is.na(heart$chol_bp_ratio)), "\n")
cat(sprintf("chol/trestbps ratio -> min=%.3f  max=%.3f  mean=%.3f\n",
            min(heart$chol_bp_ratio, na.rm = TRUE),
            max(heart$chol_bp_ratio, na.rm = TRUE),
            mean(heart$chol_bp_ratio, na.rm = TRUE)))


# -------------------------------------------------------------
# TASK 3: Loop-Based vs Vectorized Data Cleaning
# -------------------------------------------------------------

# Replicate the raw BP vector to obtain a benchmark large enough for a
# meaningful system.time() comparison.
big_bp <- rep(heart$trestbps, times = 3000)   # ~909,000 elements
m <- length(big_bp)
cat("\n--- Task 3: Loop vs Vectorized (n =", m, ") ---\n")

# Loop-based detection of invalid/outlier BP values
loop_time <- system.time({
  flag_loop <- logical(m)
  for (i in 1:m) {
    v <- big_bp[i]
    if (is.na(v)) {
      flag_loop[i] <- NA
    } else if (v < 0 || v > 250) {
      flag_loop[i] <- TRUE
    } else {
      flag_loop[i] <- FALSE
    }
  }
})

# Vectorized detection of the same invalid/outlier BP values
vec_time <- system.time({
  flag_vec <- ifelse(is.na(big_bp), NA,
               ifelse(big_bp < 0 | big_bp > 250, TRUE, FALSE))
})

cat("Loop-based invalid count      :", sum(flag_loop, na.rm = TRUE), "\n")
cat("Vectorized invalid count      :", sum(flag_vec,  na.rm = TRUE), "\n")
cat("Loop execution time (s)       :", loop_time["elapsed"], "\n")
cat("Vectorized execution time (s) :", vec_time["elapsed"], "\n")
cat("Speed-up (loop / vectorized)  :", round(loop_time["elapsed"] / vec_time["elapsed"], 1), "x\n")


# -------------------------------------------------------------
# TASK 4: Validate the Cleaned Data
# -------------------------------------------------------------

cat("\n--- Task 4: Validation of cleaned BP ---\n")
cat("Missing BP count:", sum(is.na(heart$trestbps_clean)), "\n")
cat("Minimum BP      :", min(heart$trestbps_clean, na.rm = TRUE), "\n")
cat("Maximum BP      :", max(heart$trestbps_clean, na.rm = TRUE), "\n")
cat("Mean BP         :", round(mean(heart$trestbps_clean, na.rm = TRUE), 2), "\n")
cat("Median BP       :", median(heart$trestbps_clean, na.rm = TRUE), "\n")
cat("Any negative BP remaining? :", any(heart$trestbps_clean < 0, na.rm = TRUE), "\n")
cat("Any BP > 250 remaining?    :", any(heart$trestbps_clean > 250, na.rm = TRUE), "\n")


# -------------------------------------------------------------
# EXPORT: Deliverable
# -------------------------------------------------------------

heart$trestbps <- heart$trestbps_clean
heart$trestbps_clean <- NULL
write.csv(heart, "cleaned_heart_data.csv", row.names = FALSE)
cat("\nCleaned dataset exported as cleaned_heart_data.csv\n")
cat("\n=== LAB 3 SCRIPT COMPLETE ===\n")


## Lab 4: Advanced Missing Data Handling
Dataset: UCI Adult / Census Income Dataset.
Tasks: distinguishing `NA`, `NULL`, `NaN` and blank strings; missing-data treatment strategy; a custom median-imputation function; before/after missingness comparison using `naniar`; and validation with `skimr::skim()`.

In [ ]:
%%R
# =============================================================
# Lab 4: Advanced Missing Data Handling
# Topic  : NA, NULL, NaN, Missing-Value Detection and Imputation
# Dataset: UCI Adult / Census Income Dataset (adult_income.csv, 48,842 records)
# =============================================================

if (!require(naniar))  install.packages("naniar")
if (!require(skimr))   install.packages("skimr")
library(naniar)
library(skimr)

# -------------------------------------------------------------
# SETUP: Import the dataset and inject additional missing-data scenarios
# -------------------------------------------------------------

# The raw UCI Adult dataset already encodes missing categorical values with
# "?". These are treated as NA at import time (a standard first step for
# this dataset).
adult <- read.csv("https://raw.githubusercontent.com/jbrownlee/Datasets/master/adult-all.csv",
                   header = FALSE, stringsAsFactors = FALSE, na.strings = c("?"))
colnames(adult) <- c("age","workclass","fnlwgt","education","education_num",
                      "marital_status","occupation","relationship","race","sex",
                      "capital_gain","capital_loss","hours_per_week","native_country","income")

cat("Rows:", nrow(adult), " Columns:", ncol(adult), "\n")
cat("Natural missingness already present in the raw data (via '?'):\n")
cat("  workclass      :", sum(is.na(adult$workclass)), "\n")
cat("  occupation     :", sum(is.na(adult$occupation)), "\n")
cat("  native_country :", sum(is.na(adult$native_country)), "\n")

# Deterministic row positions (reproducible on every run) used to introduce
# additional NA, blank-string, NaN and impossible-value scenarios on top of
# the dataset's natural missingness.
age_999_idx       <- c(100, 8100, 16100, 24100, 32100, 40100)   # impossible age
workclass_na_idx  <- c(200, 8200, 16200, 24200, 32200, 40200)   # extra NA
country_blank_idx <- c(300, 8300, 16300, 24300, 32300, 40300)   # blank string ""
hours_nan_idx     <- c(400, 8400, 16400, 24400, 32400, 40400)   # NaN (undefined computed value)

adult$age[age_999_idx]                <- 999
adult$workclass[workclass_na_idx]     <- NA
adult$native_country[country_blank_idx] <- ""
adult$hours_per_week[hours_nan_idx]   <- NaN


# -------------------------------------------------------------
# TASK 1: Identify Different Forms of Missing/Invalid Data
# -------------------------------------------------------------

cat("\n--- Task 1: NA / NaN / NULL / blank / impossible-value demo ---\n")

# NA
cat("is.na()  on age right now      :", sum(is.na(adult$age)),
    "(0 expected -- the 999 sentinel is a real number, not yet NA)\n")

# NaN (a subset of is.na() in R, but distinct under is.nan())
cat("is.nan() on hours_per_week     :", sum(is.nan(adult$hours_per_week)), "\n")

# NULL: represents the absence of an R object, not a value inside a cell
missing_object <- NULL
cat("is.null(missing_object)        :", is.null(missing_object), "\n")
cat("is.null(adult$workclass)       :", is.null(adult$workclass),
    " (a data-frame column is never NULL, even when full of NA)\n")

# Blank strings
cat('Blank strings (x == "") in native_country:',
    sum(adult$native_country == "", na.rm = TRUE), "\n")

# Impossible numeric values
cat("Impossible age values (age == 999):", sum(adult$age == 999, na.rm = TRUE), "\n")

# Variable-wise missing-value summary (naniar)
selected_vars <- c("age", "workclass", "education_num", "occupation",
                    "hours_per_week", "native_country", "income")

cat("\n--- naniar::miss_var_summary() (selected variables, BEFORE cleaning) ---\n")
print(miss_var_summary(adult[selected_vars]))


# -------------------------------------------------------------
# TASK 2: Missing-Data Treatment Strategy
# -------------------------------------------------------------

# a) Convert impossible numeric values to NA
adult$age[adult$age == 999] <- NA
cat("\nage == 999 rows converted to NA:", length(age_999_idx),
    " -> total age NA now:", sum(is.na(adult$age)), "\n")

# b) Replace blank categorical values with "Unknown"
char_cols <- names(adult)[sapply(adult, is.character)]
blank_replaced <- 0
for (col in char_cols) {
  blanks <- which(adult[[col]] == "")
  blank_replaced <- blank_replaced + length(blanks)
  adult[[col]][blanks] <- "Unknown"
}
cat("Blank string cells replaced with 'Unknown':", blank_replaced, "\n")

# c) Remove observations containing unrecoverable NaN values BEFORE
#    generic median imputation runs (NaN represents an undefined computed
#    result, not a legitimately missing input -> not safe to impute).
rows_before <- nrow(adult)
adult <- adult[!is.nan(adult$hours_per_week), ]
cat("Rows removed for unrecoverable NaN in hours_per_week:",
    rows_before - nrow(adult), "\n")

# d) Custom median-imputation function
impute_median <- function(x) {
  if (!is.numeric(x)) stop("impute_median() expects a numeric vector.")
  med <- median(x, na.rm = TRUE)
  x[is.na(x)] <- med
  return(x)
}

numeric_targets <- c("age", "hours_per_week", "capital_gain", "capital_loss")
cat("\n--- Task 2d: impute_median() report ---\n")
for (col in numeric_targets) {
  before_na <- sum(is.na(adult[[col]]))
  adult[[col]] <- impute_median(adult[[col]])
  after_na <- sum(is.na(adult[[col]]))
  cat(sprintf("  %-15s | missing before: %5d | missing after: %d\n", col, before_na, after_na))
}

# e) complete.cases() on the selected variables
cat("\ncomplete.cases() on selected vars:", sum(complete.cases(adult[selected_vars])),
    "/", nrow(adult), "complete\n")


# -------------------------------------------------------------
# TASK 3: Custom Imputation Function (see impute_median() above)
# -------------------------------------------------------------
# impute_median(x): accepts a numeric vector, finds the median of the valid
# observations and replaces every missing entry with that median.


# -------------------------------------------------------------
# TASK 4: Missingness Before vs After Cleaning
# -------------------------------------------------------------

cat("\n--- naniar::miss_var_summary() (selected variables, AFTER cleaning) ---\n")
after_summary <- miss_var_summary(adult[selected_vars])
print(after_summary)

png("missingness_comparison.png", width = 900, height = 600, res = 120)
gg_miss_var(adult[selected_vars])
dev.off()
cat("\nMissingness visualization saved as missingness_comparison.png\n")


# -------------------------------------------------------------
# TASK 5: Validate the Cleaned Dataset
# -------------------------------------------------------------

cat("\n--- Task 5: Validation (skimr::skim) ---\n")
print(skim(adult[c("age", "education_num", "hours_per_week",
                    "capital_gain", "capital_loss")]))

cat("\nVerification checks:\n")
cat("  Any age == 999 remaining?          :", any(adult$age == 999), "\n")
cat("  Any blank '' remaining (char cols) :",
    any(sapply(adult[char_cols], function(x) any(x == ""))), "\n")
cat("  NA remaining in numeric targets    :\n")
print(sapply(adult[numeric_targets], function(x) sum(is.na(x))))
cat("  Final row count:", nrow(adult), "\n")


# -------------------------------------------------------------
# EXPORT: Deliverable
# -------------------------------------------------------------

write.csv(adult, "cleaned_adult_data.csv", row.names = FALSE)
cat("\nCleaned dataset exported as cleaned_adult_data.csv\n")
cat("\n=== LAB 4 SCRIPT COMPLETE ===\n")


### Task 4 (Lab 4) — Missingness Visualization
`naniar::gg_miss_var()` bar chart comparing missing/invalid counts per variable before and after cleaning.

In [ ]:
from IPython.display import Image

# Display the missingness visualization saved by the R script above
Image('missingness_comparison.png')

### Deliverables produced by this notebook
- `cleaned_heart_data.csv` — Lab 3 cleaned dataset
- `cleaned_adult_data.csv` — Lab 4 cleaned dataset
- `missingness_comparison.png` — Lab 4 missingness visualization

Download them from the Colab file browser (or add `files.download(...)` calls) before submitting.